# Hyperdimensional Computing (HDC) for Digit Identification
This notebook implements a simple HDC pipeline:
1. **Encoding**: Mapping pixels to hyperspace.
2. **Training**: Bundling vectors into class prototypes.
3. **Inference**: Cosine similarity for identification.

In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
from tqdm import tqdm

In [2]:
# Hyperparameters
D = 10000  # Dimensions
LEVELS = 256  # Grayscale intensity levels
BATCH_SIZE = 100

## 1. Data Preparation
We'll use MNIST as our benchmark for number identification.

In [3]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: torch.flatten(x))])

train_set = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)

test_set = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

100%|██████████| 9.91M/9.91M [00:01<00:00, 7.47MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 206kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 2.27MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 1.91MB/s]


Using device: cuda


## 2. HDC Foundation: Basis Vectors
We create:
- **Position Vectors**: Random bipolar vectors for each pixel (784 total).
- **Level Vectors**: Correlated vectors representing pixel intensity.

In [4]:
# Random bipolar vectors (-1, 1)
def generate_bipolar_vectors(num, dim):
    return torch.sign(torch.randn(num, dim)).to(device)

# Position vectors for each pixel index
proj_vectors = generate_bipolar_vectors(784, D)

# Level vectors with ID-level coding (similarity correlates to value distance)
level_vectors = torch.zeros(LEVELS, D).to(device)
level_vectors[0] = generate_bipolar_vectors(1, D)
for i in range(1, LEVELS):
    level_vectors[i] = level_vectors[i-1].clone()
    # Flip a small number of bits to maintain correlation with previous level
    idx = torch.randperm(D)[:D // (2 * LEVELS)]
    level_vectors[i][idx] *= -1

## 3. The Encoder
We bind the pixel position with its intensity value and bundle them all together.

In [5]:
def encode(images):
    # images shape: [Batch, 784]
    # Map pixel values (0.0 - 1.0) to level indices (0 - 255)
    pixel_indices = (images * (LEVELS - 1)).long()
    
    # Get level vectors for all pixels: [Batch, 784, D]
    levels = level_vectors[pixel_indices]
    
    # Bind (Multiply in bipolar) with Position Vectors
    # [Batch, 784, D] * [784, D] -> [Batch, 784, D]
    bound = levels * proj_vectors
    
    # Bundle (Sum) across all pixels
    # [Batch, D]
    query_vectors = torch.sum(bound, dim=1)
    return torch.sign(query_vectors) # Bipolar quantization

## 4. Training (Creating Class Prototypes)

In [6]:
class_vectors = torch.zeros(10, D).to(device)

print("Training: Bundling class vectors...")
for images, labels in tqdm(train_loader):
    images, labels = images.to(device), labels.to(device)
    encoded_images = encode(images)
    
    for i in range(10):
        # Accumulate encoded vectors for each digit class
        class_vectors[i] += torch.sum(encoded_images[labels == i], dim=0)

# Final Class Vectors
class_vectors = torch.sign(class_vectors)

Training: Bundling class vectors...


100%|██████████| 600/600 [07:38<00:00,  1.31it/s]


## 5. Inference (Evaluation)

In [7]:
correct = 0
total = 0

print("Testing: Comparing query vectors to class prototypes...")
with torch.no_grad():
    for images, labels in tqdm(test_loader):
        images, labels = images.to(device), labels.to(device)
        q = encode(images)
        
        # Cosine similarity is equivalent to Dot Product for bipolar vectors
        distances = torch.matmul(q, class_vectors.T)
        predictions = torch.argmax(distances, dim=1)
        
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

print(f"\nAccuracy: {100 * correct / total:.2f}%")

Testing: Comparing query vectors to class prototypes...


100%|██████████| 100/100 [01:14<00:00,  1.33it/s]


Accuracy: 78.96%
